# Build the FI/macro-only positioning panel

Recomputes the hedge fund positioning variables using **only funds classified Fixed income / rates RV
or Global macro** (`Data/fund_strategies.csv`, written by `build_fund_classification.ipynb`) and merges
them onto the existing bond panel. Bond-side variables are untouched, so
`monetary_policy_induced_position_fimacro.csv` is a drop-in input for the robustness specifications.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyodbc

ROOT = Path.cwd() if (Path.cwd() / "Data").exists() else Path.cwd() / "build"
DATA = ROOT / "Data"

strat = pd.read_csv(DATA / "fund_strategies.csv")
keep_leis = tuple(strat.loc[strat["strategy"].isin(
    ["Fixed income / rates RV", "Global macro"]), "lei"])
print(len(keep_leis), "FI/macro funds")

cnxn = pyodbc.connect('DSN=Hermes_DSN', autocommit=True)

df = pd.read_csv('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\bond_timeseries_v2.csv')
df['collateral_country'] = df['ISIN'].str[:2]
securities = tuple(df[df['collateral_country'].isin(['DE', 'IT'])]['ISIN'].unique())

In [ ]:
query = f"""

SELECT s.business_date,
security_isin as isin,
borrower_id as fund_id,
sum(nominal_euro) as long_vol

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'
AND nominal_ccy IN ('EUR')
AND central_clearing = 'non-cleared'
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY business_date, isin, borrower_id

"""

df_long = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT s.business_date,
security_isin as isin,
lender_id as fund_id,
sum(nominal_euro) as short_vol

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'
AND nominal_ccy IN ('EUR')
AND central_clearing = 'non-cleared'
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY business_date, isin, lender_id

"""

df_short = pd.read_sql_query(query, cnxn)

In [ ]:
fb = df_long.merge(df_short, on=['business_date', 'isin', 'fund_id'], how='outer')
fb[['long_vol', 'short_vol']] = fb[['long_vol', 'short_vol']].fillna(0)
fb = fb[fb['fund_id'].isin(keep_leis)]
pos = (fb.groupby(['business_date', 'isin'])[['long_vol', 'short_vol']].sum().reset_index())
pos['business_date'] = pd.to_datetime(pos['business_date'])
pos['net_pos'] = (pos['long_vol'] - pos['short_vol']) / 1e9

In [ ]:
out = pd.read_csv('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\monetary_policy_induced_position.csv')
out['business_date'] = pd.to_datetime(out['business_date'])

out = out.rename(columns={'hf_involved': 'hf_involved_all',
                          'hf_intensity_pre': 'hf_intensity_all'})
hf_cols = ['net_pos', 'abs_net', 'net_pos_scaled', 'hf_intensity_long',
           'hf_intensity_short', 'prev_net_pos', 'hf_involved_long',
           'hf_involved_short', 'borrowing_volume', 'lending_volume', 'daily_net_change',
           'delta_intensity', 'is_long_pre', 'is_short_pre']
out = out.drop(columns=[c for c in hf_cols if c in out.columns])
out = out.merge(pos[['business_date', 'isin', 'net_pos']], on=['business_date', 'isin'], how='left')
out['net_pos'] = out['net_pos'].fillna(0)

In [ ]:
out = out.sort_values(['isin', 'business_date'])
out['abs_net'] = (out['net_pos'].abs() / out['amt_issued']) * 100

out['hf_intensity_pre'] = (out.groupby('isin')['abs_net']
    .apply(lambda s: s.rolling(window=5, min_periods=3).mean().shift(1))
    .reset_index(level=0, drop=True))

out['net_pos_scaled'] = (out['net_pos'] / out['amt_issued']) * 100
hf_roll_signed = (out.groupby('isin')['net_pos_scaled']
    .apply(lambda s: s.rolling(window=5, min_periods=3).mean().shift(1))
    .reset_index(level=0, drop=True))
out['hf_intensity_long'] = np.where(hf_roll_signed > 0, out['hf_intensity_pre'], 0)
out['hf_intensity_short'] = np.where(hf_roll_signed < 0, out['hf_intensity_pre'], 0)
out['hf_intensity_pre'] = out['hf_intensity_pre'].fillna(0)

out.loc[out['hf_intensity_pre'] > 0, 'hf_involved'] = 1
out['hf_involved'] = out['hf_involved'].fillna(0)

out.to_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\monetary_policy_induced_position_fimacro.csv')
print(out['hf_involved'].mean().round(3), "share of bond-days with FI/macro positioning")